# IndoML T2 — E8 (Kaggle 2×T4, free): MP-SENet fine-tune, real run — λ=0 (GPU 0) and λ=5 ASR-feature loss (GPU 1)
DNS-pretrained MP-SENet fine-tuned in fp32 on the realistic simulation (8,000 mixtures; isolated val noise from train clusters + ESC-50; event SNR −20…+5 dB), 5 epochs, OneCycle LR 2e-4, bs 2, 1 s segments. Best epoch picked by held-out-cluster SI-SDR; final leak-free eval (held-out synthetic + ALL natural, blends 1.0/0.9). 7.5 h safety limit per run.

Smoke reference (1,000 mix × 2 ep): held-out 14.39 → 14.74 dB, ΔWER syn 0.62 → 2.36, nat −0.53 → +0.06, est 14.23 → 15.53.

Inputs: `indoml-validation`, `indoml-t2-ckpt`. **GPU T4 ×2**, Internet ON, HF_TOKEN. ~6–8 h. Outputs: `mpsenet_ft_lam0.pt`, `mpsenet_ft_lam5.pt`.

In [ ]:
!rm -rf /kaggle/working/ner && git clone -q --depth 1 https://github.com/nagshubhadip/noise-event-detection-and-removal /kaggle/working/ner
!pip -q install jiwer sentencepiece webrtcvad-wheels librosa 2>&1 | tail -1
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /kaggle/working/esc50.zip && unzip -q -o /kaggle/working/esc50.zip -d /kaggle/working/ && ls /kaggle/working/ESC-50-master | head
# ---- MP-SENet (MIT) ----
!rm -rf /kaggle/working/mpsenet && git clone -q --depth 1 https://github.com/yxlu-0102/MP-SENet /kaggle/working/mpsenet
!pip -q install einops 2>&1 | tail -1

In [ ]:
import sys, os, glob, json, time, random, numpy as np, torch, soundfile as sf, librosa
from pathlib import Path
VAL = Path('/kaggle/input/datasets/avin55/indoml-validation/validation')
M = Path('/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular')
d = (M/'data.py').read_text()
d = d.replace('Path("/kaggle/input/indoml-validation"),', 'Path("/kaggle/input/datasets/avin55/indoml-validation/validation"), Path("/kaggle/input/indoml-validation"),')
d = d.replace('cid = (r.get("clip_id") or r.get("id")', 'cid = (r.get("segmentFileName") or r.get("clip_id") or r.get("id")')
(M/'data.py').write_text(d); sys.path.insert(0, str(M))
from kaggle_secrets import UserSecretsClient
HF = UserSecretsClient().get_secret("HF_TOKEN"); os.environ["HF_TOKEN"] = HF
import config, data, evaluate
import model as model_mod
from train import train_on_sim
from inference import enhance, Transcriber
from jiwer import wer as jwer
config.ensure_dirs()
val_dir, items = data.resolve_validation()
meta = {m['segmentFileName'][:-4]: m for m in json.load(open(VAL/'validationMetadata.json', encoding='utf-8'))}
print('validation clips', len(items))
import torch.nn.functional as F

In [ ]:
import types, sys
sys.modules.setdefault('pesq', types.SimpleNamespace(pesq=lambda *a, **k: 0.0))   # PESQ not needed (no metric discriminator)
sys.path.insert(0, '/kaggle/working/mpsenet')
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
h = AttrDict(json.load(open('/kaggle/working/mpsenet/best_ckpt/config.json')))
dev = torch.device('cuda')
def load_mp(path='/kaggle/working/mpsenet/best_ckpt/g_best_dns'):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)['generator']); return m.eval()
def mp_forward(m, wav):                       # wav [B, T] (RMS-normalised) -> enhanced [B, T]
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    amp_g, pha_g, com_g = m(amp, pha)
    return mag_pha_istft(amp_g, pha_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (amp_g, pha_g, com_g)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev)
    W, H = int(win * sr), int(hop * sr)
    if n <= W:
        y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); wsum = torch.zeros(n, device=dev)
        fade = torch.ones(W, device=dev); r = W - H; fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev)
        s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = torch.nn.functional.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]
            L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; wsum[s:e] += f
            if e == n: break
            s += H
        y = y / wsum.clamp_min(1e-6)
    y = (y / nf).cpu().numpy()
    return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
mp = load_mp(); print('MP-SENet params (M):', round(sum(p.numel() for p in mp.parameters()) / 1e6, 2))

In [ ]:
# Isolated validation noise events + fingerprint clustering
SR = 16000
NV = Path('/kaggle/working/bank/val_noise'); NV.mkdir(parents=True, exist_ok=True)
segs, fps, owner = [], [], []
for v in items:
    if not v['syn']: continue
    y = data.read_audio(v['path']); c = data.read_audio(v['ref']); L = min(len(y), len(c)); n = (y[:L] - c[:L]).astype(np.float32)
    for k, e in enumerate(meta.get(v['id'], {}).get('NoiseSubCategoryTimeStamp', [])):
        a, b = max(0, int((float(e['start']) - 0.02) * SR)), min(L, int((float(e['end']) + 0.02) * SR))
        if b - a < int(0.3 * SR) or np.abs(n[a:b]).max() < 1e-4: continue
        seg = n[a:b]; p = NV / f"{v['id'][-40:]}_{k}.wav"; sf.write(p, seg, SR)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=48, n_fft=512, hop_length=160) + 1e-10)
        f = librosa.util.fix_length(mel, size=48, axis=1).ravel(); f = (f - f.mean()) / (f.std() + 1e-6)
        segs.append({'path': str(p), 'dur': (b - a) / SR}); fps.append(f / np.linalg.norm(f)); owner.append(v['id'])
F = np.stack(fps); cents, lab = [], np.zeros(len(F), int)
for i, f in enumerate(F):
    if cents:
        s = np.stack(cents) @ f; j = int(s.argmax())
        if s[j] > 0.95: lab[i] = j; continue
    cents.append(f); lab[i] = len(cents) - 1
ncl = len(cents); rng = np.random.default_rng(7)
print('noise events', len(F), '| clusters', ncl)
for frac in [0.35, 0.45, 0.55]:
    held = set(rng.permutation(ncl)[:int(frac * ncl)].tolist())
    clip_cl = {}
    for o, l in zip(owner, lab): clip_cl.setdefault(o, set()).add(int(l))
    HELD_CLIPS = [o for o, s in clip_cl.items() if s <= held]
    if len(HELD_CLIPS) >= 80: break
train_noise = [s for s, l in zip(segs, lab) if int(l) not in held]
print(f'held-out clusters {len(held)}/{ncl} | held-out eval clips {len(HELD_CLIPS)} | train noise segments {len(train_noise)}')

In [ ]:
# ESC-50 noise segments (active part, 0.3-2.5 s), resampled to 16 kHz
E = Path('/kaggle/working/ESC-50-master'); NE = Path('/kaggle/working/bank/esc50'); NE.mkdir(parents=True, exist_ok=True)
import pandas as pd
esc = pd.read_csv(E/'meta/esc50.csv'); esc_noise = []
for _, r in esc.iterrows():
    w, _ = librosa.load(E/'audio'/r.filename, sr=SR, mono=True)
    w, _ = librosa.effects.trim(w, top_db=30)
    if len(w) < int(0.3 * SR): continue
    w = w[:int(2.5 * SR)].astype(np.float32); p = NE / r.filename; sf.write(p, w, SR)
    esc_noise.append({'path': str(p), 'dur': len(w) / SR, 'cat': r.category})
print('ESC-50 segments', len(esc_noise), '| categories', esc.category.nunique())

In [ ]:
SCRIPT = r'''
import sys, os, json, time, glob, argparse, types, numpy as np, torch, torch.nn.functional as F
ap = argparse.ArgumentParser(); ap.add_argument("--lam", type=float); ap.add_argument("--epochs", type=int, default=5)
ap.add_argument("--out"); ap.add_argument("--bs", type=int, default=2); ap.add_argument("--seg", type=int, default=16000)
ap.add_argument("--lr", type=float, default=2e-4); ap.add_argument("--hours", type=float, default=7.5); a = ap.parse_args()
M = "/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular"; sys.path.insert(0, M)
sys.modules.setdefault("pesq", types.SimpleNamespace(pesq=lambda *x, **k: 0.0)); sys.path.append("/kaggle/working/mpsenet")   # append: MP-SENet also has an inference.py
HF = os.environ["HF_TOKEN"]
import config, data, evaluate
import model as model_mod
from inference import Transcriber
from jiwer import wer as jwer
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
torch.manual_seed(0); np.random.seed(0)
h = AttrDict(json.load(open("/kaggle/working/mpsenet/best_ckpt/config.json"))); dev = torch.device("cuda")
def load_mp(path="/kaggle/working/mpsenet/best_ckpt/g_best_dns"):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)["generator"]); return m
def mp_forward(m, wav):
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    a, p, c = m(amp, pha); return mag_pha_istft(a, p, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (a, p, c)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev); W, H = int(win*sr), int(hop*sr)
    if n <= W: y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); ws = torch.zeros(n, device=dev); r = W - H
        fade = torch.ones(W, device=dev); fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev); s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = F.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]; L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; ws[s:e] += f
            if e == n: break
            s += H
        y = y / ws.clamp_min(1e-6)
    y = (y / nf).cpu().numpy(); return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
def post(x, y):
    ri, ro = np.sqrt((x**2).mean()), np.sqrt((y**2).mean()); z = y * (ri / ro) if ro > 1e-8 else y
    pk = np.abs(z).max(); return (z * (0.99 / pk) if pk > 0.99 else z).astype(np.float32)
records = [json.loads(l) for l in open(config.MANIFEST)]
tdl, _, _, _ = data.build_dataloaders(records, a.bs)
encoder = enc_fwd = None
if a.lam > 0: _, encoder, _ = model_mod.core_model(HF); enc_fwd = type(encoder).forward.__wrapped__
val_dir, items = data.resolve_validation(); byid = {v["id"]: v for v in items}
H = json.load(open("/kaggle/working/held_clips.json")); X = {i: data.read_audio(byid[i]["path"]) for i in H}; REF = {i: data.read_audio(byid[i]["ref"]) for i in H}
def heldout_sisdr(m):
    m.eval(); s = float(np.mean([evaluate.si_sdr(REF[i], post(X[i], enhance_mp(m, X[i]))) for i in H])); m.train(); return s
m = load_mp(); m.train(); opt = torch.optim.AdamW(m.parameters(), lr=a.lr, betas=(0.8, 0.99))
total = a.epochs * len(tdl); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=total, pct_start=0.05)
best = heldout_sisdr(m); print(f"[lam={a.lam}] zero-shot held-out SI-SDR {best:.2f}", flush=True); torch.save({"generator": m.state_dict()}, a.out)
t0 = time.time(); step = 0; stop = False
for ep in range(a.epochs):
    run = None
    for b in tdl:
        mix, clean = b["mix"].to(dev), b["clean"].to(dev)
        if mix.shape[-1] > a.seg: st = np.random.randint(0, mix.shape[-1] - a.seg); mix, clean = mix[:, st:st+a.seg], clean[:, st:st+a.seg]
        nf = torch.sqrt(mix.shape[-1] / (mix.pow(2).sum(-1, keepdim=True) + 1e-9)); mix, clean = mix * nf, clean * nf
        c_amp, c_pha, c_com = mag_pha_stft(clean, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        y, (a_g, p_g, co_g) = mp_forward(m, mix); y = y[:, :clean.shape[-1]]
        _, _, co_h = mag_pha_stft(y, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
        ip, gd, iaf = phase_losses(c_pha, p_g)
        loss = 0.9*F.mse_loss(c_amp, a_g) + 0.3*(ip+gd+iaf) + 0.2*F.mse_loss(c_com, co_g) + 0.2*F.mse_loss(co_g, co_h) + 0.2*F.l1_loss(clean, y)
        if a.lam > 0:
            with torch.no_grad(): fc = enc_fwd(encoder, clean / nf); fc = F.layer_norm(fc.float(), fc.shape[-1:])
            fe = enc_fwd(encoder, y / nf); fe = F.layer_norm(fe.float(), fe.shape[-1:]); loss = loss + a.lam * (fe - fc).abs().mean()
        opt.zero_grad(set_to_none=True); loss.backward(); gn = torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
        if torch.isfinite(gn): opt.step()
        sched.step(); step += 1; run = loss.item() if run is None else 0.99 * run + 0.01 * loss.item()
        if step % 500 == 0: print(f"[lam={a.lam}] ep{ep} step{step}/{total} loss {run:.4f} | {(time.time()-t0)/3600:.2f} h", flush=True)
        if time.time() - t0 > a.hours * 3600: stop = True; break
    s = heldout_sisdr(m); print(f"[lam={a.lam}] epoch {ep} held-out SI-SDR {s:.2f} (best {best:.2f})", flush=True)
    if s > best: best = s; torch.save({"generator": m.state_dict()}, a.out)
    if stop: print("time limit reached", flush=True); break
# final leak-free eval of the best checkpoint: held-out synthetic + ALL natural
del encoder; torch.cuda.empty_cache()
m = load_mp(a.out).eval(); noisy = json.load(open(os.environ["NOISY_ASR"])); asr = Transcriber(HF)
NAT = [v["id"] for v in items if not v["syn"]]; GT = {i: evaluate.normalize(byid[i]["gt"]) for i in H + NAT}
for i in NAT: X[i] = data.read_audio(byid[i]["path"])
def pooled(ids, hyp): ids = [i for i in ids if GT[i]]; return jwer([GT[i] for i in ids], [evaluate.normalize(hyp.get(i, "")) or "@" for i in ids])
ENH = {i: enhance_mp(m, X[i]) for i in H + NAT}
for bl in [1.0, 0.9]:
    out = {i: post(X[i], bl * ENH[i] + (1 - bl) * X[i]) for i in H + NAT}
    si = float(np.mean([evaluate.si_sdr(REF[i], out[i]) for i in H])); hyp = {}; ids = H + NAT
    for k in range(0, len(ids), 16):
        for i, t in zip(ids[k:k+16], asr.transcribe_many([out[i] for i in ids[k:k+16]])): hyp[i] = t
    ds, dn = 100 * (pooled(H, noisy) - pooled(H, hyp)), 100 * (pooled(NAT, noisy) - pooled(NAT, hyp))
    print("[RESULT]", json.dumps(dict(lam=a.lam, blend=bl, si_sdr_heldout=round(si, 2), dWER_syn_heldout=round(ds, 2), dWER_nat_all=round(dn, 2), combined_est=round(si + 0.32*ds + 0.68*dn, 2))), flush=True)
'''

In [ ]:
# Bigger realistic simulation for the real run (shared by both GPUs)
import gc
clean_idx, _ = data.build_banks(6000, hf_token=HF)
noise_pool = train_noise + esc_noise
config.SIM.update(snr_db=(-20.0, 5.0), events=(1, 4), max_noise_sec=2.5, min_noise_sec=0.3)
for f in list(config.MIX_DIR.glob('*.wav')) + list(config.REF_DIR.glob('*.wav')): f.unlink()
records = data.simulate_dataset(clean_idx, noise_pool, 8000)
json.dump(HELD_CLIPS, open('/kaggle/working/held_clips.json', 'w'))
open('/kaggle/working/mp_ft.py', 'w').write(SCRIPT)
del mp; gc.collect(); torch.cuda.empty_cache()
print('mixtures', len(records), '| held-out clips', len(HELD_CLIPS))

In [ ]:
import subprocess
NOISY = glob.glob('/kaggle/input/**/noisy_asr_val.json', recursive=True)[0]
def go(lam, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), NOISY_ASR=NOISY, PYTORCH_ALLOC_CONF='expandable_segments:True'); log = f'/kaggle/working/mpft_lam{lam}.log'
    return subprocess.Popen(['python', '/kaggle/working/mp_ft.py', '--lam', str(lam), '--epochs', '5', '--out', f'/kaggle/working/mpsenet_ft_lam{lam}.pt'],
                            stdout=open(log, 'w'), stderr=subprocess.STDOUT, env=env), log
jobs = {0: go(0, 0), 5: go(5, 1)}
while any(p.poll() is None for p, _ in jobs.values()):
    time.sleep(900)
    for lam, (p, f) in jobs.items():
        L = [l for l in open(f).read().splitlines() if l.startswith('[lam')]
        print(f"[lam={lam} | {'running' if p.poll() is None else 'EXIT ' + str(p.returncode)}]", L[-1] if L else '', flush=True)
for lam, (p, f) in jobs.items():
    print(f'===== lam {lam} (exit {p.returncode})')
    print('\n'.join(l for l in open(f).read().splitlines() if ('held-out' in l) or l.startswith(('[RESULT]', 'Traceback', 'time limit')) or 'Error' in l))